# GATE28 evaluation: stock Qwen3.5-9B vs feyninc/sqrl-9b (bf16, vLLM, thinking on)

Runtime → Change runtime type → **A100 or G4 if available, else L4**. Then **Runtime → Run all**.
Data pack: from the private repo if a `GH_TOKEN` secret exists, otherwise the notebook asks you to upload `gate28_pack.tar.gz`.
Results go to Google Drive: `MyDrive/zalo_eval/GATE28/`. Both models run with identical settings; nothing is quantized.

In [ ]:
MODELS = [('stock', 'Qwen/Qwen3.5-9B'), ('sqrl9b', 'feyninc/sqrl-9b')]
OUT = '/content/drive/MyDrive/zalo_eval/GATE28'
THINKING, MAX_TOKENS, CTX = True, 8192, 32768

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, subprocess, json, time
from google.colab import drive, userdata
drive.mount('/content/drive')
try:
    os.environ['GH_TOKEN'] = userdata.get('GH_TOKEN')
except Exception:
    os.environ['GH_TOKEN'] = ''
if not os.environ['GH_TOKEN'] and not os.path.exists('/content/gate28_pack.tar.gz'):
    from google.colab import files
    print('No GH_TOKEN secret: upload gate28_pack.tar.gz')
    up = files.upload()
    os.replace(list(up)[0], '/content/gate28_pack.tar.gz') if list(up)[0] != 'gate28_pack.tar.gz' else os.replace('gate28_pack.tar.gz', '/content/gate28_pack.tar.gz')
os.makedirs(OUT, exist_ok=True)
mem = int(subprocess.check_output(['nvidia-smi','--query-gpu=memory.total','--format=csv,noheader,nounits']).decode().split()[0])
SEQS = 28 if mem > 40000 else 12      # L4 (24 GB): fewer parallel sequences
print('GPU MB', mem, '-> parallel questions', SEQS)

In [ ]:
%%bash
set -e
cd /content
rm -rf zalo-vecna && git clone -q --depth 1 -b harness-h3b https://github.com/JimmyK300/zalo-vecna.git
if [ ! -f /content/gate28_pack.tar.gz ]; then
ASSET=$(curl -s -H "Authorization: Bearer $GH_TOKEN" https://api.github.com/repos/JimmyK300/zalo-ft1-data/releases/tags/gate28-pack-v1 | python -c "import sys,json;print([a['url'] for a in json.load(sys.stdin)['assets'] if a['name']=='gate28_pack.tar.gz'][0])")
curl -sL -H "Authorization: Bearer $GH_TOKEN" -H "Accept: application/octet-stream" "$ASSET" -o /content/gate28_pack.tar.gz
fi
mkdir -p /content/pack && tar -xzf /content/gate28_pack.tar.gz -C /content/pack
ls /content/pack /content/pack/cache
pip -q install vllm duckdb==2.0.0 requests 2>&1 | tail -2
python -c "import vllm, duckdb, transformers; print('vllm', vllm.__version__, 'duckdb', duckdb.__version__, 'transformers', transformers.__version__)" 

In [ ]:
import requests, shutil
def serve(model):
    log = open(f'/content/vllm_{model.split("/")[-1]}.log', 'w')
    p = subprocess.Popen(['vllm', 'serve', model, '--dtype', 'bfloat16', '--max-model-len', str(CTX),
                          '--gpu-memory-utilization', '0.92', '--max-num-seqs', str(SEQS), '--seed', '47',
                          '--enable-prefix-caching', '--limit-mm-per-prompt', '{"image":0,"video":0}'],
                         stdout=log, stderr=subprocess.STDOUT)
    for _ in range(240):
        time.sleep(5)
        try:
            if requests.get('http://127.0.0.1:8000/v1/models', timeout=5).ok:
                print('server ready:', model); return p
        except Exception:
            pass
        if p.poll() is not None:
            break
    print(open(log.name).read()[-3000:]); raise RuntimeError('vLLM did not start')

for tag, model in MODELS:
    t0 = time.time()
    p = serve(model)
    try:
        cmd = ['python', '-m', 'harness3.raw_eval', '--pack', '/content/pack', '--model', model, '--tag', tag,
               '--out', '/content/out', '--max-tokens', str(MAX_TOKENS), '--ctx', str(CTX), '--workers', str(SEQS)]
        if THINKING: cmd.append('--thinking')
        subprocess.run(cmd, cwd='/content/zalo-vecna', check=True)
    finally:
        p.terminate(); p.wait(60)
    for f in os.listdir('/content/out'):
        shutil.copy(f'/content/out/{f}', OUT)
    print(f'{tag} done in {(time.time()-t0)/60:.1f} min; copied to {OUT}')
    time.sleep(10)

In [ ]:
import collections
for tag, _ in MODELS:
    rs = [json.loads(l) for l in open(f'/content/out/{tag}.jsonl', encoding='utf8')]
    print(tag, len(rs), collections.Counter(r['stop'] for r in rs))
print('Done. Tell Claude: GATE28 finished.')